In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import logging
import warnings
warnings.filterwarnings('ignore')

# --- LOGLAMA AYARLARI ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- PROFESYONEL MIMARI PARAMETRELERI ---
OUTPUT_DIR = "processed_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_FILE = os.path.join(OUTPUT_DIR, 'train_processed.csv')
TEST_FILE = os.path.join(OUTPUT_DIR, 'test_processed.csv')

def get_data():
    if not os.path.exists('BRFSS2024.csv'):
        print("Sistemde CSV bulunamadi. XPT dosyasi donusturuluyor...")
        df_raw = pd.read_sas('LLCP2024.XPT')
        df_raw.to_csv('BRFSS2024.csv', index=False)
        return df_raw
    return pd.read_csv('BRFSS2024.csv')

def optimize_memory(df):
    for col in df.columns:
        if df[col].dtype == 'float64':
            df[col] = df[col].astype(np.float32)
    df.columns = [c.upper() for c in df.columns]
    return df

def handle_outliers_iqr(df, columns):
    """
    Proje Kilavuzu 1. Hafta: Aykiri Degerlerin Tespiti ve Yonetimi
    Surekli degiskenler icin IQR yontemi ile aykiri degerleri baskilar (Clipping/Winsorization).
    """
    print("\n🔍 Aykiri Degerler (Outliers) IQR Yontemi ile Baskilaniyor...")
    df_clean = df.copy()
    for col in columns:
        if col in df_clean.columns:
            Q1 = df_clean[col].quantile(0.25)
            Q3 = df_clean[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR

            # Aykiri degerleri sinir degerlere esitliyoruz (Baskilama)
            df_clean[col] = np.clip(df_clean[col], lower_bound, upper_bound)
    return df_clean

def feature_engineering_grup1(df, sleep_col):
    """
    Grup 1 Ozel Gorevi: BMI, Fiziksel Aktivite ve Uyku Duzeni etkilesimleri.
    """
    # Egzersiz Eksikligi Riski
    if '_BMI5' in df.columns and 'EXERANY2' in df.columns:
        df['LACK_OF_EXERCISE'] = df['EXERANY2'].replace({1: 0, 2: 1})
        df['METABOLIC_RISK_IDX'] = df['_BMI5'] * df['LACK_OF_EXERCISE']

    # Uyku Duzeni Riski (Grup 1 Odak Noktasi)
    # Ideal uyku 7-8 saat kabul edilir. Bunun disindakiler risk faktoru olarak kodlanir.
    if sleep_col in df.columns:
        df['POOR_SLEEP_PATTERN'] = df[sleep_col].apply(
            lambda x: 1 if (pd.notna(x) and (x < 6 or x > 9)) else 0
        )
    return df

def smart_knn_imputation(df_scaled, imputer, dataset_name="Veri"):
    print(f"\n🚀 [{dataset_name}] KNN Imputasyonu Baslatiliyor...")
    missing_mask = df_scaled.isna().any(axis=1)
    df_complete = df_scaled[~missing_mask]
    df_missing = df_scaled[missing_mask]

    if len(df_missing) == 0:
        return df_scaled

    fit_sample_size = min(1000, len(df_complete))
    fit_sample = df_complete.sample(n=fit_sample_size, random_state=42)
    imputer.fit(fit_sample)

    imputed_array = imputer.transform(df_missing)
    df_missing_imputed = pd.DataFrame(imputed_array, columns=df_scaled.columns, index=df_missing.index)

    df_final = pd.concat([df_complete, df_missing_imputed]).sort_index()
    print(f"   ✅ [{dataset_name}] Imputasyon Tamamlandi!")
    return df_final

# ================= ANA BORU HATTI (PIPELINE) =================

if os.path.exists(TRAIN_FILE) and os.path.exists(TEST_FILE):
    print(f"✅ Islenmis veri setleri '{OUTPUT_DIR}' klasorunde mevcut.")
else:
    print("Veri Isleme Boru Hatti (Pipeline) Tam Kapasite Baslatiliyor...")

    df = get_data()
    df = optimize_memory(df)

    SLEEP_COL = 'SLEPTIM2' if 'SLEPTIM2' in df.columns else 'SLEPTIM1'
    INCOME_COL = 'INCOME3' if 'INCOME3' in df.columns else 'INCOME2'

    # Kilavuz Tablo 2'de belirtilen temel degiskenlere uyum
    columns_to_keep = ['DIABETE4', '_BMI5', '_AGEG5YR', '_RFHLTH', 'EXERANY2',
                       SLEEP_COL, '_SMOKER3', '_PHYS14D', INCOME_COL, '_EDUCAG']
    df = df[[c for c in columns_to_keep if c in df.columns]]

    # 1. Labeling (Hedef Degisken Ikili Formata Cevriliyor)
    df = df.dropna(subset=['DIABETE4'])
    df = df[df['DIABETE4'].isin([1, 2, 3, 4])]
    df['DIABETE4'] = df['DIABETE4'].replace({2: 1, 4: 1, 3: 0}).astype(np.int8)

    # 2. Temel Veri Temizligi ve Veri Tipi Duzeltmeleri
    if '_BMI5' in df.columns:
        df['_BMI5'] = df['_BMI5'] / 100.0

    # Anket kodlarini (Bilmiyorum/Reddedildi) NaN yapiyoruz
    survey_null_codes = [7, 9, 77, 99, 777, 999]
    features = df.columns.drop('DIABETE4')
    df[features] = df[features].replace(survey_null_codes, np.nan)

    # 3. Aykiri Deger Tespiti ve Baskilama (Surekli Degiskenler Icin)
    continuous_features = ['_BMI5', SLEEP_COL, '_PHYS14D']
    df = handle_outliers_iqr(df, continuous_features)

    # 4. Oznitelik Muhendisligi (Grup 1: Metabolik Saglik, BMI, Uyku, Egzersiz)
    df = feature_engineering_grup1(df, SLEEP_COL)

    # 5. Train/Test Split (Veriden Sizmayi Onlemek Icin Imputasyondan Once Yapilmali)
    print("\nEgitim ve Test setleri ayriliyor (%80 Egitim - %20 Test)...")
    X = df.drop('DIABETE4', axis=1)
    y = df['DIABETE4']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

    X_train, X_test = X_train.reset_index(drop=True), X_test.reset_index(drop=True)
    y_train, y_test = y_train.reset_index(drop=True), y_test.reset_index(drop=True)

    # 6. Feature Scaling (Oznitelik Olceklendirme)
    # Not: Karar agaclari icin sart olmasa da Veri Mimari rolu geregi yapiliyor.
    print("Veri olceklendiriliyor (StandardScaler)...")
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

    # 7. KNN Imputasyonu
    imputer = KNNImputer(n_neighbors=5, weights='distance')
    X_train_imputed = smart_knn_imputation(X_train_scaled, imputer, dataset_name="Egitim Seti")
    X_test_imputed = smart_knn_imputation(X_test_scaled, imputer, dataset_name="Test Seti")

    # 8. Sinif Dengesizligi (SMOTE)
    print("\n⚖️ Sinif dengesizligi SMOTE ile gideriliyor...")
    smote = SMOTE(random_state=42)
    X_train_final, y_train_final = smote.fit_resample(X_train_imputed, y_train)

    print("\n💾 Islenmis veriler CSV olarak kaydediliyor...")
    train_processed = pd.concat([X_train_final, y_train_final], axis=1)
    test_processed = pd.concat([X_test_imputed, y_test], axis=1)

    train_processed.to_csv(TRAIN_FILE, index=False)
    test_processed.to_csv(TEST_FILE, index=False)

    print(f"🎉 ISLEM TAMAMLANDI! Model ekibi (Makine Ogrenmesi Muhendisleri) icin veriler hazir.")

Veri Isleme Boru Hatti (Pipeline) Tam Kapasite Baslatiliyor...
Sistemde CSV bulunamadi. XPT dosyasi donusturuluyor...

🔍 Aykiri Degerler (Outliers) IQR Yontemi ile Baskilaniyor...

Egitim ve Test setleri ayriliyor (%80 Egitim - %20 Test)...
Veri olceklendiriliyor (StandardScaler)...

🚀 [Egitim Seti] KNN Imputasyonu Baslatiliyor...
   ✅ [Egitim Seti] Imputasyon Tamamlandi!

🚀 [Test Seti] KNN Imputasyonu Baslatiliyor...
   ✅ [Test Seti] Imputasyon Tamamlandi!

⚖️ Sinif dengesizligi SMOTE ile gideriliyor...

💾 Islenmis veriler CSV olarak kaydediliyor...
🎉 ISLEM TAMAMLANDI! Model ekibi (Makine Ogrenmesi Muhendisleri) icin veriler hazir.
